# Preparation of a FEMTIC inversion input files: data and mesh 

Example with the AusLAMP South Australia Dataset

In [ ]:
import sys
sys.path.append('../../src')
import femticPy

# 1. Prepare inversion data file

In [ ]:
# the file will be written in the input_data directory
inversion = femticPy.DataGen(survey = 'auslamp', outdir = './inversion')

In [ ]:
# Loading the data and the data coordinates
inversion.read_MTdata('./input_data/edi_files/')
inversion.read_MTdata_coordinates('./input_data', 'coord_auslampSA')

In [ ]:
## Data to be inverted for
inversion.invert_Z   = True
inversion.invert_VTF = False
inversion.invert_PT  = False

In [ ]:
# load the 3 files needed to create the mesh
inversion.topography = './input_data/topography.dat'
inversion.bathymetry = './input_data/bathymetry.dat'
inversion.coast_line = './input_data/coast_line.dat'

In [ ]:
# we center the data to a anchor point (center of the data set) which will also be the center of the future mesh
inversion.center_data()
inversion.anchor

In [ ]:
# we define the analysis domain, the extent of the total mesh including padding
inversion.analysis_domain = [[-1000.0, 1000.0],
                                [-1000.0, 1000.0],
                                [-1000.0, 1000.0]]
inversion.plot_data_loc(plot_ids = False, zoom_core=False)
inversion.plot_coast_line()
inversion.plot_topo_bathy()

In [ ]:
inversion.mt_coords

In [ ]:
# Set up error floors
# x% of (srqt(Zxy^2 + Zyx^2))
inversion.error_floor_Z = [0.2, 0.05, 0.05, 0.2]  # [Zxx, Zxy, Zyx, Zyy]

In [ ]:
subsampling = 2
inversion.freqs[::subsampling]

In [ ]:
inversion.write_observe(write = True, freq_bandwidth = [0.0001,150], subsampling = subsampling)

In [ ]:
# Inspect the data before inversion
import MTtools as MTtools
MTtools.plot_Z(inversion,
               xlim = [1,5])

# 2. Prepare inversion control file

In [ ]:
# Define inversion parameters:

#TRADE_OFF_PARAM
TRADE_OFF_PARAM = [3]

# Max number of iterations
ITERATION = 10

# NUM_THREADS
NUM_THREADS = 4

# CONVERGE
CONVERGE = 0.2

# output directory
outdir = 'inversion'

In [ ]:
inversion.write_inversion_control(TRADE_OFF_PARAM = TRADE_OFF_PARAM,
                                    ITERATION = ITERATION,
                                    NUM_THREADS = NUM_THREADS,
                                    CONVERGE = CONVERGE)

# 3. Prepare mesh files

In [ ]:
# output directory
outdir = 'meshGen/files'

In [ ]:
# create meshGen object
mesh01 = femticPy.MeshGen(inversion.survey,
                         inversion.analysis_domain, 
                         inversion.center, 
                         inversion.mt_coords,
                         inversion.nRx_Z,
                         outdir)

In [ ]:
mesh01.sea = True
mesh01.sea

In [ ]:
# Step 1: 

# Define the mesh disretization, sphere centered on the mesh center 
    # core area: 650km sphere radius (isotropic) / max cell size : 50km (anisotropic in z)
    # padding: 2000km sphere radius (isotropic), max cell size: 500km (isotropic)

mesh01.center = [0.0, 0.0, 0.0]
mesh01.rotation = -45
mesh01.ellipsoids_control = [2,
                     [650.0, 50.0, 0.35, 0.0, 0.0],
                     [2000.0, 500.0, 0.0, 0.0, 0.0]]

mesh01.coast_line = './input_data/coast_line.dat'

mesh01.ellipsoids_observing_sites =  [6, 
                                      1.0, 0.5,
                                      5.0, 1.0,
                                     50.0, 15.0,
                                      100.0, 25.0,
                                      150.0,50.0,
                                     200.0, 100.0]

In [ ]:
# Step 6: Refine
# this step refines the tetras around the MT sites

mesh01.ellipsoids_obs_sites = [4,
                    [1.0, 0.5, 0.0] ,          
                    [10.0, 5.0, 0.0] ,           
                    [100.0, 25.0, 0.0],         
                    [200.0, 100.0, 0.0],]


mesh01.ellipsoids_mtr = [2,
                    [400.0, 50.0, 0.0, 0.1, 0.1],
                    [2000.0, 500.0, 0.0, 0.0, 0.0]]

In [ ]:
# Step 7: Write mesh to FEMTIC format
# the attr file will determine how tetras (elements) will be combined into cells (model parameters)

mesh01.ellipsoids_resistivity_attr = [2,
                    [400.0,       50.0,  0.0,  0.0],
                    [2000.0,  500.0,  0.0,  0.0]] 


mesh01.ellipsoids_resistivity_attr_sites = [3,                        
                    [10.0, 2.0],
                    [50.0, 10.0],
                    [150.0, 25.0]]

In [ ]:
# Define the starting model resistivities (in ohm.m)
mesh01.resistivity_starting_model = 100

In [ ]:
mesh01.write_inputs()

In [ ]:
# number of regions (air, land, sea) 
mesh01.region_attributes

# 4. Run `create_mesh.sh`

the `create_mesh.sh` script call the `meshGen.sh` pipeline, that sequentially runs all the steps required to create the mesh. Depending on the size of the mesh this can take long (~ 10 min). 

"Step 4" in `meshGen.sh` needs to be modified to your problem:

```
#-----------------------"
#-----   STEP 4    -----"
#-----------------------"

echo "STEP 4... "
/scratch3/sei029/femtic/makeTetraMesh -stp 4
sed -i "$(( $(wc -l < output.poly) ))s/.*/3/" output.poly
echo "1   0.0 0.0 -40.0  10 1e9" >> output.poly
echo "2  -500.0 -500.0  1.0  20 1e9" >> output.poly
echo "3   0.0 0.0  40.0  30 1e9" >> output.poly
echo "   done!"
sleep 1
```

In this step, 3 lines are added to the `output.poly` file. They correspond to:

`"region_attribute   x   y   z   region_number   1e9"`

x, y, z (in km) should indicate at which location a sample of each region can be found. z is positive downwards.

The region numbers have to match with the ones in `resistivity_attr.dat`

```
3
10 1.0e+9 -1 1
20 0.250  -1 1
30 100.0   9 0
```

10 is for air, 20 for sea and 30 for earth (with associated resistivities)

"Step 5" in `meshGen.sh` calls a python script that fixes a bug that has been appearing sometimes. This is a temporary fix while understanding why a new region number (31 or 32) sometimes appears during step 5. 

```
echo "fix output.1.ele if necessary... "
module load python
python fix_output1.py
sleep 1
echo "   done!"
```

To check the size of the mesh, check the `resistivity_block_iter0.dat` file. The first number is the number of elements, the second is te number of cells.

```
>head resistivity_block_iter0.dat 
   1453333    206736
         0         0
....
```

# 5. Visualise the mesh using paraview (before running the inversion)

Setting up the ellipsoids is cumbersome and takes some trials to obtain a mesh that looks ok, in terms of cells discretization and mesh size. 

If the shell script has run properly, the final files to be used for the inversion should be:
- resistivity_block_iter0.dat
- mesh.dat



The output files that can be copied over and are useful for visualization are:

- output.6.femtic.vtk
   - to check the model parameter discretization (the cells), use the blockSerial attribute of the .vtk file
- triangles_with_height.vtk
- coastLine_fine.vtk
- coastLine_rough.vtk

The last 3 files are in km, and a transform filter has to be applied in paraview to be visualized with `output.6.femtic.vtk` (which is m)

# 6. Running the inversion using `run_inversion.sh`